# Classifier-free guidance

In [1]:
import numpy as np

def cfg_combine(eps_uncond, eps_cond, w):
    return eps_uncond + w * (eps_cond - eps_uncond)

def conditional_direction(eps_uncond, eps_cond):
    return eps_cond - eps_uncond

def guidance_projection(eps_uncond, eps_cond, w):
    d = conditional_direction(eps_uncond, eps_cond).flatten()
    n = np.linalg.norm(d)
    if n == 0:
        return 0.0
    guided = cfg_combine(eps_uncond, eps_cond, w).flatten()
    return float(np.dot(guided, d / n))

def guidance_alignment(eps_uncond, eps_cond, w):
    d = conditional_direction(eps_uncond, eps_cond).flatten()
    g = cfg_combine(eps_uncond, eps_cond, w).flatten()
    nd, ng = np.linalg.norm(d), np.linalg.norm(g)
    if nd == 0 or ng == 0:
        return 0.0
    return float(np.dot(g, d) / (nd * ng))

def alignment_curve(eps_uncond, eps_cond, ws):
    return np.array([guidance_alignment(eps_uncond, eps_cond, w) for w in ws])

In [2]:
u = np.array([1.0, 0.0]); c = np.array([1.0, 1.0])
print(cfg_combine(u, c, 0.0))                       # [1. 0.]
print(cfg_combine(u, c, 1.0))                       # [1. 1.]
print(guidance_projection(u, c, 3.0))               # 3.0
print(round(guidance_alignment(u, c, 0.0), 3))      # 0.0
print(round(guidance_alignment(u, c, 1.0), 3))      # 0.707
print(np.round(alignment_curve(u, c, [0, 1, 2, 4, 8]), 3))  # [0. 0.707 0.894 0.970 0.992]

rng = np.random.RandomState(0)
eu = rng.randn(4, 16, 16); ec = rng.randn(4, 16, 16)
print(cfg_combine(eu, ec, 7.5).shape)               # (4, 16, 16)
print(np.allclose(cfg_combine(eu, ec, 7.5), eu + 7.5 * (ec - eu)))

[1. 0.]
[1. 1.]
3.0
0.0
0.707
[0.    0.707 0.894 0.97  0.992]
(4, 16, 16)
True
